# SkinScope AI — Dermatological ML Pipeline (Google Colab)

This notebook reproduces SkinScope AI's **Chromatic ML Classifier** (`HSV` + `YCbCr` feature extraction + region-calibrated Random Forest & Rule-Based models) on `skinscope_dermatological_dataset.csv`.

- **Dataset:** 300 labeled samples across `skin`, `palm`, and `eye` (`Healthy appearance`, `Jaundice`, `Erythema / Burn / Inflammation`)
- **Models:** Random Forest Classifier + SkinScope Region Rule Engine

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset (or upload skinscope_dermatological_dataset.csv in Colab)
URL = 'https://raw.githubusercontent.com/STEVEN-197/Skin-Scope-Ai/main/ml-colab/skinscope_dermatological_dataset.csv'
df = pd.read_csv(URL)
print('Dataset shape:', df.shape)
df.head()

In [ ]:
# Encode region & target severity
le_region = LabelEncoder()
df['region_enc'] = le_region.fit_transform(df['region'])

features = ['region_enc', 'avgR', 'avgG', 'avgB', 'avgH', 'avgS', 'avgV', 'yellowRatio', 'redRatio', 'darkRatio', 'skinTissueRatio', 'scleraRatio']
X = df[features]
y = df['severity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=150, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f'Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n')
print(classification_report(y_test, y_pred))

In [ ]:
# Plot feature importance
imp = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)
imp.plot(kind='barh', figsize=(8, 4), title='SkinScope AI — ML Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Test on a new sample (e.g., Eye with 7.5% yellow sclera)
sample = pd.DataFrame([{
    'region_enc': le_region.transform(['eye'])[0],
    'avgR': 195, 'avgG': 200, 'avgB': 80, 'avgH': 52, 'avgS': 45, 'avgV': 78,
    'yellowRatio': 7.5, 'redRatio': 0.5, 'darkRatio': 3.0, 'skinTissueRatio': 35.0, 'scleraRatio': 14.0
}])
print('Predicted severity:', clf.predict(sample)[0])